# Aula 01 — Fundamentos e Panorama da Ciência de Dados · Exercícios

**Unidade 1 — Introdução à Ciência de Dados e Python** · ver também: [conteúdo](conteudo.ipynb) · [atividades](atividades.ipynb)

## 🗂️ Microdados — ENEM 2025 / RN

Conjuntos de dados utilizados no exemplo da aula: microdados de resultados do ENEM 2025 no RN e catálogo de escolas. Baixe os dois arquivos para executar o exemplo.

---
## Catálogo Escola RN

> Arquivo salvo em `microdados/catalogo_escolas_rn.csv`.

---
## Microdados ENEM 2025 RN

> Arquivo salvo em `microdados/RESULTADOS_2025_RN.csv`.

## 💻 Exemplo Prático — Ranking das Escolas do RN (ENEM 2025)

Código completo do estudo de caso apresentado no livro da aula: leitura dos microdados, cálculo das médias, ranking e geração da página HTML interativa.

---
## Exemplo — Código do Ranking das Escolas (main.py)

### Exemplo Prático — Ranking das Escolas do RN no ENEM 2025

Este exemplo percorre o ciclo completo visto no livro da aula: lê os microdados, calcula médias por aluno e por escola, junta os dados com o catálogo de nomes e gera uma página HTML interativa com o ranking.

#### Estrutura do projeto

```text
projeto/
├── main.py                      ← script principal (anexo nesta seção)
├── template_escolas.html        ← template da página final (anexo)
└── microdados/
    ├── RESULTADOS_2025_RN.csv   ← microdados ENEM 2025 RN (seção Microdados)
    └── catalogo_escolas_rn.csv  ← catálogo de escolas (seção Microdados)
```

#### Como executar

1.  Baixe os quatro arquivos e organize-os como acima (crie a pasta `microdados/`);
2.  Execute `python main.py` (Python 3.10+, sem dependências externas);
3.  O script processa ~30 mil registros e abre `escolas_rn.html` no navegador.

#### Pontos de atenção no código

- **Formatos heterogêneos:** os resultados usam separador `;` e codificação `latin1`; o catálogo usa vírgula e `utf-8` — situação comum ao integrar fontes reais.
- **Valores ausentes:** alunos faltosos têm notas vazias; o código converte para `None` e calcula a média apenas das notas válidas.
- **Junção por chave:** `CO_ESCOLA` liga os dois arquivos — o mesmo papel de um *join* em SQL ou `merge` no Pandas.
- **Separação código/apresentação:** os dados são injetados no template HTML via `json.dumps`, substituindo o marcador `__DADOS_ESCOLAS__`.

#### main.py

In [ ]:
import csv
import json
import webbrowser
from pathlib import Path
from statistics import mean

BASE_DIR = Path.cwd()  # no notebook: cwd = pasta da aula (__file__ não existe)
RESULTADOS_CSV = BASE_DIR / "microdados" / "RESULTADOS_2025_RN.csv"
CATALOGO_CSV = BASE_DIR / "microdados" / "catalogo_escolas_rn.csv"
OUTPUT_HTML = BASE_DIR / "escolas_rn.html"

TIPO_DEPENDENCIA = {
    "1": "Federal",
    "2": "Estadual",
    "3": "Municipal",
    "4": "Privada",
}

NOTA_COLUNAS = {
    "NU_NOTA_CN": "cn",
    "NU_NOTA_CH": "ch",
    "NU_NOTA_LC": "lc",
    "NU_NOTA_MT": "mt",
    "NU_NOTA_REDACAO": "redacao",
}

In [ ]:
def carregar_catalogo_nomes():
    nomes = {}
    with open(CATALOGO_CSV, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            nomes[row["CO_ESCOLA"]] = row["NO_ESCOLA"]
    return nomes

In [ ]:
def carregar_escolas():
    nomes = carregar_catalogo_nomes()
    escolas = {}

    with open(RESULTADOS_CSV, encoding="latin1") as f:
        for row in csv.DictReader(f, delimiter=";"):
            codigo = row["CO_ESCOLA"]
            if not codigo:
                continue

            escola = escolas.setdefault(codigo, {
                "codigo": codigo,
                "municipio": row["NO_MUNICIPIO_ESC"],
                "tipo": TIPO_DEPENDENCIA.get(row["TP_DEPENDENCIA_ADM_ESC"], "-"),
                "alunos": [],
            })

            notas_aluno = {}
            for coluna, chave in NOTA_COLUNAS.items():
                valor = row[coluna]
                notas_aluno[chave] = round(float(valor), 1) if valor else None

            validas = [v for v in notas_aluno.values() if v is not None]
            notas_aluno["geral"] = round(mean(validas), 1) if validas else None
            escola["alunos"].append(notas_aluno)

    lista = []
    for escola in escolas.values():
        # aluno sem nenhuma nota valida vai para o final
        alunos_ordenados = sorted(
            escola["alunos"],
            key=lambda a: a["geral"] if a["geral"] is not None else -1,
            reverse=True,
        )
        lista.append({
            "codigo": escola["codigo"],
            "nome": nomes.get(escola["codigo"], ""),
            "municipio": escola["municipio"],
            "tipo": escola["tipo"],
            "alunos": alunos_ordenados,
        })

    return lista

In [ ]:
def gerar_html(escolas):
    template = (BASE_DIR / "template_escolas.html").read_text(encoding="utf-8")
    html = template.replace("__DADOS_ESCOLAS__", json.dumps(escolas, ensure_ascii=False))
    OUTPUT_HTML.write_text(html, encoding="utf-8")

In [ ]:
def main():
    escolas = carregar_escolas()
    gerar_html(escolas)
    print(f"{len(escolas)} escolas processadas.")
    print(f"Arquivo gerado: {OUTPUT_HTML}")
    webbrowser.open(OUTPUT_HTML.as_uri())

In [ ]:
if __name__ == "__main__":
    main()

---
## Exemplo — Páginas HTML Análise Escolas

> Pacote `Exemplo_Analise_Escolas.zip` extraído nesta pasta: `escolas_rn.html`, `template_escolas.html`.